# Detoxification via attractor traversal

Companion to the paper's §*Attractors for traversals / toxicity*. Build toxic and neutral mean-activation prototypes from Llama-2-7B on the ParaDetox corpus, then subtract the toxic prototype from a middle attention layer during generation (negative `strength`) to detoxify completions. Toxicity and content retention are scored with ParlAI's `OffensiveLanguageClassifier` and ROUGE-1.

## Setup

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from parlai.utils.safety import OffensiveLanguageClassifier


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf", torch_dtype=torch.bfloat16,
).to(device)
model = model.eval()


## Data

In [ ]:
df = pd.read_csv("paradetox.tsv", sep='\t', header=0)
df.drop(['neutral2', 'neutral3'], axis=1, inplace=True)

In [ ]:
prompts = []
toxic = []

for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    prompts.append(row['toxic'])
    toxic.append(1)
    prompts.append(row['neutral1'])
    toxic.append(0)

prompts = np.array(prompts)
toxic = np.array(toxic)

keep_idx = random.sample(list(range(len(prompts))), 600)
prompts = prompts[keep_idx]
toxic = toxic[keep_idx]

# Sort so neutral (0) comes first, toxic (1) last — makes the cosine
# heatmap below show two clean class blocks.
order = np.argsort(toxic)
toxic = toxic[order]
prompts = prompts[order]


## Prototype extraction

Hook every self-attention output, run one forward pass per prompt, mean-pool across the sequence, then group by class to form one mean-activation prototype per (label, layer).

In [ ]:
activations = {}
def get_activation(name):
    def hook(model, input, output):
        if name in activations:
            activations[name].append(output[0])
        else:
            activations[name] = [output[0]]
    return hook


hooks = []
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.self_attn.register_forward_hook(get_activation(i)))


In [ ]:
hidden_states = []
activations = {}

pbar = tqdm(enumerate(prompts), total=len(prompts))
for i, p in pbar:

    pbar.set_description('cur prompt length (chars): %d' %(len(p)))
    
    with torch.no_grad():
        inputs = tokenizer(p, return_tensors="pt").input_ids
        outputs = model.forward(inputs.to(device))
        
    for k in activations.keys():
        activations[k] = torch.cat(activations[k], dim=1).detach().cpu()
        activations[k] =  torch.mean(activations[k], dim=-2) #activations[k][..., -1, :]


    hidden_states.append(copy.deepcopy(activations))
    activations = {}

In [ ]:
target_layers = list(range(len(model.model.layers)))
all_prompts_prototypes = hidden_states

prototypes = {l: {} for l in [0, 1]}
for target_layer in target_layers:
    for l in [0, 1]:
        cur_prototypes = []
        for i in range(len(all_prompts_prototypes)):
            if toxic[i] != l:
                continue
            cur_prototypes.append(all_prompts_prototypes[i][target_layer])
        cur_prototypes = torch.stack(cur_prototypes, dim=0)
        prototypes[l][target_layer] = torch.mean(cur_prototypes, dim=0)


## Prototype similarity

Pairwise cosine between every prompt's layer-16 activation. Prompts sort cleanly into neutral vs toxic blocks — the attractor structure the steering intervention below will subtract from.

In [ ]:
target_layer = 16

cs = np.zeros((len(prompts), len(prompts)))
cs_func = nn.CosineSimilarity(dim=-1)
for i in range(len(prompts)):
    for j in range(len(prompts)):
        cs[i, j] = cs_func(all_prompts_prototypes[i][target_layer], all_prompts_prototypes[j][target_layer])

fig, ax = plt.subplots(figsize=(5, 5))
plt.imshow(cs, cmap='Reds')
plt.xticks([i for i in range(0, len(prompts), 100)], toxic[::100], rotation=90)
plt.yticks([i for i in range(0, len(prompts), 100)], toxic[::100], rotation=0)
plt.title(f'layer {target_layer}')
plt.show()


## Steering intervention

Remove the capture hooks and install one steering hook on layer 16 that adds `strength * prototypes[1][16]` to the attention output. Negative `strength` subtracts the toxic prototype → detoxification.

In [ ]:
for h in hooks:
    h.remove()


In [ ]:
target_layer = 16
target_style = 1   # toxic prototype
strength = 0      # overridden per-iteration in the sweep below


def change_behavior(target_layer):
    def hook(model, input, output):
        output = (output[0] + strength * prototypes[target_style][target_layer].to(device), *output[1:])
        return output
    return hook


hook = model.model.layers[target_layer].self_attn.register_forward_hook(change_behavior(target_layer))


## Validation set

Random 300 toxic prompts held out for the sweep.

In [ ]:
toxic_prompts = []
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    toxic_prompts.append(row['toxic'])

toxic_prompts = np.array(toxic_prompts)
keep_idx = random.sample(list(range(len(toxic_prompts))), 300)
toxic_prompts = toxic_prompts[keep_idx]


## Detox sweep

For each strength in `[-4, -6, -8, -10, -12]`, generate a detoxed paraphrase for every validation prompt at layer 16 and score it with the ParlAI toxicity classifier (fraction classified toxic) and ROUGE-1 against the original toxic input.

In [ ]:
toxicity_classifier = OffensiveLanguageClassifier(custom_model_file='zoo:bot_adversarial_dialogue/multi_turn/model')
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

for strength in [-4, -6, -8, -10, -12]:
    post = []
    rouge = []
    for sentence in tqdm(toxic_prompts, desc=f'strength={strength}'):
        prompt = f'Original: "{sentence}"; Paraphrased: "'
        with torch.no_grad():
            inputs = tokenizer(prompt, return_tensors="pt").input_ids
            outputs = model.generate(
                inputs.to(device), max_new_tokens=32,
                do_sample=False, top_k=50, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id, use_cache=True,
            ).detach().cpu()
        output_text = (
            tokenizer.decode(outputs[0, inputs.shape[1]:])
            .replace("\n", "").replace("{", "").replace("}", "")
            .replace('"', '').strip('".')
            .replace(',,', '')
            .replace('original', '').replace('Original', '')
            .split('rewritten')[0].split('revised')[0]
            .replace('10', '')
            .split('.')[0]
        )
        ppred, prob = toxicity_classifier.contains_offensive_language(output_text)
        post.append(bool(prob > 0.9 and ppred))
        rouge.append(scorer.score(sentence, output_text)['rouge1'].fmeasure)
    print(f'strength={strength}: toxicity={np.sum(post)/len(post):.3f}, rouge={np.mean(rouge):.3f}')
